In [ ]:
import os
import re
import pandas as pd

# Списки окон обучения и прогноза
TRAIN_WINDOW_lst = [3, 5, 7, 10, 15, 20, 25, 30]
PREDICT_WINDOW_lst = [3, 5, 7, 10, 15, 20, 25, 30]

for train_window in TRAIN_WINDOW_lst:
    for predict_window in PREDICT_WINDOW_lst:
        wind = f"{train_window}_{predict_window}"

        print("\n" + "=" * 80)
        print(f"Обработка окон: обучение={train_window}, прогноз={predict_window} (wind={wind})")
        print("=" * 80)

        itog_pattern = re.compile(
            rf'^итог_средние_Результаты_(.+)_{re.escape(wind)}\.xlsx$',
            re.IGNORECASE,
        )

        # Предсказания_056_Beijing_1_453_SIR.xlsx
        pred_num_pattern = re.compile(r'^Предсказания_(\d+)_')

        # Имя выходного файла для данной пары окон
        output_name = f'Общ. итог среднее {wind}.xlsx'

        model_dfs = {}

        # Ищем все итоговые файлы для текущей пары окон
        for fname in os.listdir('.'):
            m = itog_pattern.match(fname)
            if not m:
                continue

            model_name = m.group(1)  # SIRS, SIR, IMM_EKF_0.85 и др.
            print(f'Обрабатываю модель: {model_name}, файл: {fname}')

            df = pd.read_excel(fname)

            if 'Имя файла' not in df.columns or 'средн % откл I' not in df.columns:
                print(f'  Пропуск файла {fname}: нет нужных столбцов')
                continue

            tmp = df[['Имя файла', 'средн % откл I']].copy()

            # Выделяем номер предсказания
            pred_nums = []
            for full_name in tmp['Имя файла']:
                full_name = str(full_name)
                pm = pred_num_pattern.match(full_name)
                if pm:
                    pred_num = pm.group(1)  # '056', '058', ...
                else:
                    pred_num = None
                pred_nums.append(pred_num)

            tmp['Номер предсказания'] = pred_nums

            col_name = f'средн % откл I - {model_name}'
            tmp.rename(columns={'средн % откл I': col_name}, inplace=True)

            tmp = tmp[['Номер предсказания', col_name]]

            model_dfs[model_name] = tmp

        if not model_dfs:
            raise RuntimeError(
                f'Не найдено ни одного файла вида "итог_средние_Результаты_МОДЕЛЬ_{wind}.xlsx"'
            )

        models = sorted(model_dfs.keys())
        result = model_dfs[models[0]]

        for model_name in models[1:]:
            result = result.merge(
                model_dfs[model_name],
                on='Номер предсказания',
                how='outer'
            )

        # Для удобства можно восстановить одно примерное имя файла, например, на основе SIRS
        if 'SIRS' in model_dfs:
            sirs_file = None
            for fname in os.listdir('.'):
                m = itog_pattern.match(fname)
                if m and m.group(1) == 'SIRS':
                    sirs_file = fname
                    break
            if sirs_file is not None:
                df_sirs = pd.read_excel(sirs_file)[['Имя файла']].copy()
                pred_nums = []
                for full_name in df_sirs['Имя файла']:
                    full_name = str(full_name)
                    pm = pred_num_pattern.match(full_name)
                    pred_nums.append(pm.group(1) if pm else None)
                df_sirs['Номер предсказания'] = pred_nums
                df_sirs = df_sirs[['Номер предсказания', 'Имя файла']].drop_duplicates()
                result = result.merge(df_sirs, on='Номер предсказания', how='left')
                result.rename(columns={'Имя файла': 'Пример имени файла (SIRS)'}, inplace=True)

        # Сохраняем итог для данной пары окон
        result.to_excel(output_name, index=False)
        print(f'Готово, файл сохранён как: {output_name}')


Обработка окон: обучение=3, прогноз=3 (wind=3_3)
Обрабатываю модель: IMM-EKF_0.6, файл: итог_средние_Результаты_IMM-EKF_0.6_3_3.xlsx
Обрабатываю модель: IMM-EKF_0.7, файл: итог_средние_Результаты_IMM-EKF_0.7_3_3.xlsx
Обрабатываю модель: IMM-EKF_0.85, файл: итог_средние_Результаты_IMM-EKF_0.85_3_3.xlsx
Обрабатываю модель: IMM-EKF_0.99, файл: итог_средние_Результаты_IMM-EKF_0.99_3_3.xlsx
Готово, файл сохранён как: Общ. итог среднее 3_3.xlsx

Обработка окон: обучение=3, прогноз=5 (wind=3_5)
Обрабатываю модель: IMM-EKF_0.6, файл: итог_средние_Результаты_IMM-EKF_0.6_3_5.xlsx
Обрабатываю модель: IMM-EKF_0.7, файл: итог_средние_Результаты_IMM-EKF_0.7_3_5.xlsx
Обрабатываю модель: IMM-EKF_0.85, файл: итог_средние_Результаты_IMM-EKF_0.85_3_5.xlsx
Обрабатываю модель: IMM-EKF_0.99, файл: итог_средние_Результаты_IMM-EKF_0.99_3_5.xlsx
Готово, файл сохранён как: Общ. итог среднее 3_5.xlsx

Обработка окон: обучение=3, прогноз=7 (wind=3_7)
Обрабатываю модель: IMM-EKF_0.6, файл: итог_средние_Результаты